# Day 36: DreamBooth – Fine‑tune Stable Diffusion on Custom Subject

**Note:** Full training requires a GPU with at least 16GB VRAM (e.g., T4, V100, A10). This notebook provides the complete pipeline; run it on Colab or a cloud GPU instance.

In [ ]:
import os
import torch
from diffusers import StableDiffusionPipeline, AutoencoderKL
from diffusers.utils import make_image_grid
from PIL import Image

## 1. Prepare dataset
Place 4–5 images of the subject (e.g., your face) in a folder `./my_subject/`. Images should be cropped to square (512x512) and show different angles/lighting.

In [ ]:
# Example: download a dummy dataset (replace with your own)
import requests
from pathlib import Path

subject_dir = Path("./my_subject")
subject_dir.mkdir(exist_ok=True)

# Download placeholder images (replace these with your own photos)
sample_urls = [
    "https://thispersondoesnotexist.com/",  # random face, but won't work directly; use own images
]
# For actual training, you must provide your own images.
print("Place your own 512x512 face images in './my_subject' folder.")

## 2. Install and run DreamBooth training script
Hugging Face provides an official training script. We'll use it with LoRA for efficiency.

In [ ]:
%%bash
# Clone the diffusers examples and install requirements
git clone https://github.com/huggingface/diffusers
cd diffusers/examples/dreambooth
pip install -r requirements.txt

In [ ]:
# Training command (run in terminal or Colab)
training_command = """
accelerate launch diffusers/examples/dreambooth/train_dreambooth_lora.py \
  --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
  --instance_data_dir="./my_subject" \
  --output_dir="./dreambooth_lora_model" \
  --instance_prompt="a photo of sks person" \
  --resolution=512 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --learning_rate=1e-4 \
  --lr_scheduler="constant" \
  --lr_warmup_steps=0 \
  --max_train_steps=500 \
  --validation_prompt="a photo of sks person in a park" \
  --validation_epochs=50 \
  --checkpointing_steps=100
"""
print("Run this command in a GPU environment:\n")
print(training_command)

## 3. Load and use the fine‑tuned LoRA model

In [ ]:
# After training, load the LoRA weights
pipe = StableDiffusionPipeline.from_pretrained("runwayml/stable-diffusion-v1-5", torch_dtype=torch.float16).to("cuda")
pipe.load_lora_weights("./dreambooth_lora_model")

# Generate images of your subject
prompts = [
    "a photo of sks person",
    "a photo of sks person wearing a hat",
    "a photo of sks person in a futuristic city"
]
for prompt in prompts:
    image = pipe(prompt, num_inference_steps=30).images[0]
    display(image)

## 4. Alternative: Use Hugging Face Spaces or Colab notebook
Many public notebooks can run DreamBooth with a few clicks. The key is to replace `instance_prompt` with a unique identifier (e.g., `sks` or `myface`) and point to your images.